# kniv-deberta-cascade — Staged Cascade Training

Starting from the existing trained teacher (encoder + POS at 0.984 accuracy), trains cascade heads stage by stage:

- **Stage 2a**: NER MLP head on Few-NERD 45K (frozen encoder+POS)
- **Stage 2b**: POS+NER alignment on UD EWT 12.5K (frozen encoder, both heads)
- **Stage 2c**: Encoder fine-tune on posner 163K (unfrozen encoder, very low LR)
- **Stage 3**: DEP head (frozen)
- **Stage 4**: CLS head (frozen)
- **Stage 5**: Joint fine-tune all heads

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch transformers datasets sentencepiece pyyaml seqeval conllu accelerate huggingface_hub

In [ ]:
# Cell 2: Clone repo + download cascade training data + mount Drive
import os

os.chdir("/content")
!git clone --depth 1 https://github.com/rustic-ai/kniv-nlp-models.git
os.chdir("/content/kniv-nlp-models")

# Mount Google Drive (teacher checkpoint)
from google.colab import drive
drive.mount("/content/drive")

# Download cascade training data from HuggingFace
from huggingface_hub import snapshot_download
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns=["prepared/kniv-deberta-cascade/*"],
    local_dir="/content/kniv-data",
)

# Also need UD EWT CoNLL-U files for dep2label vocab
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns=["ud-english-ewt/*"],
    local_dir="/content/kniv-data",
)

# Symlink data
os.makedirs("data/prepared", exist_ok=True)
for name, src in [
    ("kniv-deberta-cascade", "/content/kniv-data/prepared/kniv-deberta-cascade"),
    ("ud-english-ewt", "/content/kniv-data/ud-english-ewt"),
]:
    dst = f"data/prepared/{name}" if "cascade" in name else f"data/{name}"
    if not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

# Verify
import json
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    vocabs = json.load(f)
for k, v in vocabs.items():
    print(f"  {k}: {len(v)} labels")

teacher = "/content/drive/MyDrive/kniv-models/deberta-v3-large-nlp-en/final"
print(f"\nTeacher checkpoint: {os.path.exists(teacher)}")
print("Ready.")

In [ ]:
# Cell 3: Stage 2a — NER MLP head on Few-NERD 45K (frozen encoder+POS, 5 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2a \
    --checkpoint /content/drive/MyDrive/kniv-models/deberta-v3-large-nlp-en/final

In [ ]:
# Cell 4: Stage 2b — POS+NER alignment on UD EWT 12.5K (frozen encoder, 1 epoch)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2b \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage2a/best

In [ ]:
# Cell 5: Stage 2d — Gentle encoder fine-tune (3 epochs)
# UD EWT 12.5K + Few-NERD 20K, encoder LR=1e-7, POS LR=1e-7, NER LR=1e-5
# Loading stage2b from Drive (local outputs lost on runtime switch)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2d \
    --checkpoint /content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en/stage2b

In [ ]:
# Cell 6: Benchmark NER on external datasets
import torch, json, sys
from transformers import AutoTokenizer
from seqeval.metrics import f1_score
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def eval_ner(model, tokenizer, ner_labels, examples, map_fn=None):
    gold_all, pred_all = [], []
    for ex in examples:
        tokens, gold_tags = ex["tokens"], ex["tags"]
        enc = tokenizer(tokens, is_split_into_words=True, max_length=128,
                       padding="max_length", truncation=True, return_tensors="pt")
        with torch.no_grad():
            out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds = out["ner_logits"][0].cpu().argmax(dim=-1).tolist()
        word_ids = enc.word_ids()
        pred_tags, prev = [], None
        for j, wid in enumerate(word_ids):
            if wid is not None and wid != prev and wid < len(tokens):
                tag = ner_labels[preds[j]] if preds[j] < len(ner_labels) else "O"
                pred_tags.append(map_fn(tag) if map_fn else tag)
            prev = wid
        min_len = min(len(gold_tags), len(pred_tags))
        gold_all.append(gold_tags[:min_len])
        pred_all.append(pred_tags[:min_len])
    return f1_score(gold_all, pred_all)

# ── Load models ──
if "model" in sys.modules: del sys.modules["model"]
sys.path.insert(0, "models/kniv-deberta-cascade-large-nlp-en")
from model import MultiTaskNLPModel as CascadeModel
cascade_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en/stage2b"
cascade_model = CascadeModel.load(cascade_dir, "microsoft/deberta-v3-large").float().to(device).eval()
cascade_tok = AutoTokenizer.from_pretrained(cascade_dir)
with open("data/prepared/kniv-deberta-cascade/label_vocabs.json") as f:
    cascade_labels = json.load(f)["ner_labels"]
print("Cascade loaded")

del sys.modules["model"]
sys.path.insert(0, "models/deberta-v3-large-nlp-en")
from model import MultiTaskNLPModel as TeacherModel
teacher_dir = "/content/drive/MyDrive/kniv-models/deberta-v3-large-nlp-en/final"
teacher_model = TeacherModel.load(teacher_dir, "microsoft/deberta-v3-large").float().to(device).eval()
teacher_tok = AutoTokenizer.from_pretrained(teacher_dir)
with open(f"{teacher_dir}/label_maps.json") as f:
    teacher_labels = json.load(f)["ner_labels"]
print("Teacher loaded")

# ── Download benchmark data (pre-prepared JSON from HuggingFace) ──
onto_path = hf_hub_download("dragonscale-ai/kniv-corpus-en", "benchmarks/ontonotes5_test.json", repo_type="dataset")
conll_path = hf_hub_download("dragonscale-ai/kniv-corpus-en", "benchmarks/conll2003_test.json", repo_type="dataset")

with open(onto_path) as f:
    onto_ex = json.load(f)
with open(conll_path) as f:
    conll_ex = json.load(f)

# ── OntoNotes 5.0 (18-type, same as our scheme) ──
print(f"\n=== OntoNotes 5.0 NER ({len(onto_ex)} test, 500 sampled) ===")
c_f1 = eval_ner(cascade_model, cascade_tok, cascade_labels, onto_ex[:500])
t_f1 = eval_ner(teacher_model, teacher_tok, teacher_labels, onto_ex[:500])
print(f"  Cascade stage2b: {c_f1:.4f}")
print(f"  Original teacher: {t_f1:.4f}")
print(f"  Delta: {c_f1 - t_f1:+.4f}")

# ── CoNLL-2003 (4-type: PER/ORG/LOC/MISC) ──
KNIV_TO_4 = {
    "PERSON": "PER", "ORG": "ORG", "GPE": "LOC", "LOC": "LOC",
    "NORP": "MISC", "FAC": "MISC", "PRODUCT": "MISC", "EVENT": "MISC",
    "WORK_OF_ART": "MISC", "LAW": "MISC", "LANGUAGE": "MISC",
    "DATE": "MISC", "TIME": "MISC", "PERCENT": "MISC", "MONEY": "MISC",
    "QUANTITY": "MISC", "ORDINAL": "MISC", "CARDINAL": "MISC",
}
def map4(tag):
    if tag == "O": return "O"
    p, e = tag[:2], tag[2:]
    m = KNIV_TO_4.get(e)
    return f"{p}{m}" if m else "O"

print(f"\n=== CoNLL-2003 NER ({len(conll_ex)} test, 500 sampled) ===")
c_f1 = eval_ner(cascade_model, cascade_tok, cascade_labels, conll_ex[:500], map4)
t_f1 = eval_ner(teacher_model, teacher_tok, teacher_labels, conll_ex[:500], map4)
print(f"  Cascade stage2b: {c_f1:.4f}")
print(f"  Original teacher: {t_f1:.4f}")
print(f"  Delta: {c_f1 - t_f1:+.4f}")

del cascade_model, teacher_model
torch.cuda.empty_cache()

In [ ]:
# Cell 7a: Fix transformers version for SpanMarker compatibility
# transformers 5.0 removed is_tf_available, SpanMarker needs 4.x
# RESTART KERNEL after this cell, then run Cell 7b
!pip install -q "transformers>=4.40,<5.0" "span_marker>=1.5"
print("Installed. RESTART KERNEL now (Runtime → Restart runtime), then run Cell 7b")

In [ ]:
# Cell 7b: Re-annotate FULL NER dataset (195K) with SpanMarker + confidence weights
# Run AFTER kernel restart + Cell 7a
import os
os.chdir("/content/kniv-nlp-models")

from google.colab import drive
drive.mount("/content/drive")

import json, torch
from span_marker import SpanMarkerModel

print("Loading SpanMarker roberta-large-ontonotes5...")
sm_model = SpanMarkerModel.from_pretrained("tomaarsen/span-marker-roberta-large-ontonotes5")
sm_model = sm_model.cuda().eval()

ENTITY_TYPES = {"PERSON","NORP","FAC","ORG","GPE","LOC","PRODUCT","EVENT",
    "WORK_OF_ART","LAW","LANGUAGE","DATE","TIME","PERCENT","MONEY","QUANTITY","ORDINAL","CARDINAL"}

def reannotate_with_spanmarker(examples, batch_size=64, min_score=0.5):
    texts = [ex["text"] for ex in examples]
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_preds = sm_model.predict(batch_texts, batch_size=batch_size)
        for j, (ex, entities) in enumerate(zip(examples[i:i+batch_size], batch_preds)):
            words = ex["words"]
            bio_tags = ["O"] * len(words)
            ner_weights = [1.0] * len(words)

            text = ex["text"]
            char_pos = 0
            word_starts = []
            for wi, w in enumerate(words):
                start = text.find(w, char_pos)
                if start == -1: start = char_pos
                word_starts.append(start)
                char_pos = start + len(w)

            for ent in entities:
                if ent["score"] < min_score or ent["label"] not in ENTITY_TYPES:
                    continue
                cs, ce = ent["char_start_index"], ent["char_end_index"]
                first_word = last_word = None
                for wi, ws in enumerate(word_starts):
                    we = ws + len(words[wi])
                    if ws < ce and we > cs:
                        if first_word is None: first_word = wi
                        last_word = wi
                if first_word is not None and all(bio_tags[t] == "O" for t in range(first_word, last_word + 1)):
                    bio_tags[first_word] = f"B-{ent['label']}"
                    ner_weights[first_word] = ent["score"]
                    for t in range(first_word + 1, last_word + 1):
                        bio_tags[t] = f"I-{ent['label']}"
                        ner_weights[t] = ent["score"]

            ex["ner_tags"] = bio_tags
            ex["ner_weights"] = ner_weights
        if (i // batch_size) % 100 == 0:
            print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", flush=True)
    return examples

# Load full NER dataset (195K = 45K kniv + 150K Few-NERD)
print("\nLoading full NER dataset...")
with open("data/prepared/kniv-deberta-cascade/ner_train.json") as f:
    all_ner = json.load(f)
print(f"  Total: {len(all_ner)} examples")

# Re-annotate everything with SpanMarker
print("\nRe-annotating with SpanMarker (91.5 F1, 18 types)...")
all_ner = reannotate_with_spanmarker(all_ner)

# Save
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_train.json", "w") as f:
    json.dump(all_ner, f)
print(f"\nSaved ner_spanmarker_train.json ({len(all_ner)} examples)")

# Also re-annotate dev set for consistent evaluation
print("\nRe-annotating dev set...")
with open("data/prepared/kniv-deberta-cascade/ner_dev.json") as f:
    ner_dev = json.load(f)
ner_dev = reannotate_with_spanmarker(ner_dev)
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_dev.json", "w") as f:
    json.dump(ner_dev, f)
print(f"Saved ner_spanmarker_dev.json ({len(ner_dev)} examples)")

# Stats
from collections import Counter
counts = Counter()
high_conf = low_conf = 0
for ex in all_ner:
    for tag, w in zip(ex["ner_tags"], ex["ner_weights"]):
        if tag != "O":
            counts[tag.split("-",1)[1]] += 1
            if w >= 0.9: high_conf += 1
            else: low_conf += 1

print(f"\nEntity distribution (195K re-annotated):")
for t, c in counts.most_common(10):
    print(f"  {t}: {c}")
total_ent = high_conf + low_conf
print(f"\nConfidence: {high_conf} high (>=0.9, {high_conf*100/max(total_ent,1):.0f}%), "
      f"{low_conf} low (<0.9, {low_conf*100/max(total_ent,1):.0f}%)")

del sm_model
torch.cuda.empty_cache()
print("\nDone. Ready for NER training on 195K SpanMarker-annotated examples.")

In [ ]:
# Cell 8: Upload SpanMarker-annotated data to HuggingFace
from huggingface_hub import HfApi
api = HfApi()

for f in ["ner_spanmarker_train.json", "ner_spanmarker_dev.json"]:
    path = f"data/prepared/kniv-deberta-cascade/{f}"
    import os
    size = os.path.getsize(path) / 1024 / 1024
    print(f"Uploading {f} ({size:.1f} MB)...")
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=f"prepared/kniv-deberta-cascade/{f}",
        repo_id="dragonscale-ai/kniv-corpus-en",
        repo_type="dataset",
    )
print("Done.")

In [ ]:
# Cell 9: Restore transformers 5.x and run NER training
!pip install -q "transformers>=5.0"
print("Restored. RESTART KERNEL then run Cell 10.")

In [ ]:
# Cell 10: Train NER on 195K SpanMarker-annotated data (after kernel restart)
import os
os.chdir("/content/kniv-nlp-models")

from google.colab import drive
drive.mount("/content/drive")

!git pull

from huggingface_hub import snapshot_download
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns=["prepared/kniv-deberta-cascade/*"],
    local_dir="/content/kniv-data",
)
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns=["ud-english-ewt/*"],
    local_dir="/content/kniv-data",
)

# Symlinks (force recreate)
os.makedirs("data/prepared", exist_ok=True)
for name, src in [
    ("data/prepared/kniv-deberta-cascade", "/content/kniv-data/prepared/kniv-deberta-cascade"),
    ("data/ud-english-ewt", "/content/kniv-data/ud-english-ewt"),
]:
    if os.path.islink(name): os.unlink(name)
    if not os.path.exists(name): os.symlink(src, name)

print("Ready. Starting Stage 2e...")
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2e \
    --checkpoint /content/drive/MyDrive/kniv-models/deberta-v3-large-nlp-en/final

In [ ]:
# Cell 11: Save stage2e checkpoint, then continue training 5 more epochs
import shutil, os

# Save current best to Drive
drive_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en"
os.makedirs(drive_dir, exist_ok=True)
src = "outputs/kniv-deberta-cascade-large-nlp-en/stage2e/best"
shutil.copytree(src, f"{drive_dir}/stage2e", dirs_exist_ok=True)
print("Saved stage2e to Drive")

# Continue training from stage2e best (5 more epochs, same config)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2e \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage2e/best

In [ ]:
# Cell 12: Re-annotate UD EWT with SpanMarker NER, then unfreeze encoder tune
# Step 1: Downgrade transformers for SpanMarker
!pip install -q "transformers>=4.40,<5.0" "span_marker>=1.5"
print("Installed. RESTART KERNEL then run Cell 13.")

In [ ]:
# Cell 13: Re-annotate UD EWT with SpanMarker + run encoder fine-tune
import os, json, torch
os.chdir("/content/kniv-nlp-models")
from google.colab import drive
drive.mount("/content/drive")

# Ensure symlinks
os.makedirs("data/prepared", exist_ok=True)
for name, src in [
    ("data/prepared/kniv-deberta-cascade", "/content/kniv-data/prepared/kniv-deberta-cascade"),
    ("data/ud-english-ewt", "/content/kniv-data/ud-english-ewt"),
]:
    if os.path.islink(name): os.unlink(name)
    if not os.path.exists(name): os.symlink(src, name)

# ── Step 1: Re-annotate UD EWT with SpanMarker ──
from span_marker import SpanMarkerModel

ENTITY_TYPES = {"PERSON","NORP","FAC","ORG","GPE","LOC","PRODUCT","EVENT",
    "WORK_OF_ART","LAW","LANGUAGE","DATE","TIME","PERCENT","MONEY","QUANTITY","ORDINAL","CARDINAL"}

print("Loading SpanMarker...")
sm_model = SpanMarkerModel.from_pretrained("tomaarsen/span-marker-roberta-large-ontonotes5")
sm_model = sm_model.cuda().eval()

with open("data/prepared/kniv-deberta-cascade/ud_ner_train.json") as f:
    ud_ner = json.load(f)
print(f"UD EWT: {len(ud_ner)} examples")

# Re-annotate NER tags with SpanMarker
texts = [ex["text"] for ex in ud_ner]
for i in range(0, len(texts), 64):
    batch_preds = sm_model.predict(texts[i:i+64], batch_size=64)
    for j, (ex, entities) in enumerate(zip(ud_ner[i:i+64], batch_preds)):
        words = ex["words"]
        bio_tags = ["O"] * len(words)
        ner_weights = [1.0] * len(words)
        text = ex["text"]
        char_pos = 0
        word_starts = []
        for wi, w in enumerate(words):
            start = text.find(w, char_pos)
            if start == -1: start = char_pos
            word_starts.append(start)
            char_pos = start + len(w)
        for ent in entities:
            if ent["score"] < 0.5 or ent["label"] not in ENTITY_TYPES:
                continue
            cs, ce = ent["char_start_index"], ent["char_end_index"]
            first_word = last_word = None
            for wi, ws in enumerate(word_starts):
                we = ws + len(words[wi])
                if ws < ce and we > cs:
                    if first_word is None: first_word = wi
                    last_word = wi
            if first_word is not None and all(bio_tags[t] == "O" for t in range(first_word, last_word + 1)):
                bio_tags[first_word] = f"B-{ent['label']}"
                ner_weights[first_word] = ent["score"]
                for t in range(first_word + 1, last_word + 1):
                    bio_tags[t] = f"I-{ent['label']}"
                    ner_weights[t] = ent["score"]
        ex["ner_tags"] = bio_tags
        ex["ner_weights"] = ner_weights

with open("data/prepared/kniv-deberta-cascade/ud_spanmarker_train.json", "w") as f:
    json.dump(ud_ner, f)
print(f"Saved ud_spanmarker_train.json ({len(ud_ner)} examples)")

del sm_model
torch.cuda.empty_cache()

# ── Step 2: Restore transformers 5.x for training ──
print("\nRestoring transformers 5.x...")
os.system('pip install -q "transformers>=5.0"')
print("RESTART KERNEL then run Cell 14.")

In [ ]:
# Cell 14: Stage 2f — Encoder tune (after kernel restart)
import os
os.chdir("/content/kniv-nlp-models")
from google.colab import drive
drive.mount("/content/drive")

!git pull

# Ensure symlinks
os.makedirs("data/prepared", exist_ok=True)
for name, src in [
    ("data/prepared/kniv-deberta-cascade", "/content/kniv-data/prepared/kniv-deberta-cascade"),
    ("data/ud-english-ewt", "/content/kniv-data/ud-english-ewt"),
]:
    if os.path.islink(name): os.unlink(name)
    if not os.path.exists(name): os.symlink(src, name)

# Stage 2f: encoder tune on UD EWT with SpanMarker NER
# Loads from stage2e (first run, NER=0.800) saved on Drive
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 2f \
    --checkpoint /content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en/stage2e

In [ ]:
# Cell 15: Save stage2f to Drive, pull fix, then run DEP (Stage 3)
import shutil, os

drive_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en"
os.makedirs(drive_dir, exist_ok=True)
src = "outputs/kniv-deberta-cascade-large-nlp-en/stage2f/best"
if os.path.exists(src):
    shutil.copytree(src, f"{drive_dir}/stage2f", dirs_exist_ok=True)
    print("Saved stage2f to Drive")

os.chdir("/content/kniv-nlp-models")
os.system("git pull")

# Stage 3: DEP head (frozen encoder+POS+NER, 5 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 3 \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage2f/best

In [ ]:
# Cell 16: SRL with MLP head (Stage 3m) — loads from stage3 (pre-SRL)
import os
os.chdir("/content/kniv-nlp-models")
os.system("git pull")

# Stage 3m: SRL MLP head (frozen encoder+POS+NER+DEP, 5 epochs)
# Loads from stage3 checkpoint (DEP only, no prior SRL)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 3m \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage3/best

In [ ]:
# Cell 17: Save stage3m to Drive + generate more SRL data on full 195K
import shutil, os, json, torch

# Save SRL MLP checkpoint
drive_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en"
os.makedirs(drive_dir, exist_ok=True)
src = "outputs/kniv-deberta-cascade-large-nlp-en/stage3m/best"
if os.path.exists(src):
    shutil.copytree(src, f"{drive_dir}/stage3m", dirs_exist_ok=True)
    print("Saved stage3m to Drive")

# Generate silver SRL on full 195K NER dataset
print("\nGenerating silver SRL on full 195K...")
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "cheralathan-m/cross-lingual-srl-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
device = torch.device("cuda")
model = model.to(device).eval()
id2label = model.config.id2label

# Our SRL tag set + mapping
SRL_TAGS_SET = set()
SRL_CORE = ["ARG0", "ARG1", "ARG2", "ARG3", "ARG4"]
SRL_MODIFIERS = ["ARGM-TMP", "ARGM-LOC", "ARGM-MNR", "ARGM-CAU", "ARGM-PRP",
    "ARGM-NEG", "ARGM-ADV", "ARGM-DIR", "ARGM-DIS", "ARGM-EXT",
    "ARGM-MOD", "ARGM-PRD", "ARGM-GOL", "ARGM-COM", "ARGM-REC"]
SRL_TAGS = ["O", "V"] + [f"{p}-{r}" for r in SRL_CORE + SRL_MODIFIERS for p in ["B", "I"]]
SRL_TAGS_SET = set(SRL_TAGS)

def map_srl_label(label):
    if label == "O": return "O"
    if label in ("B-V", "I-V"): return "V"
    if label in SRL_TAGS_SET: return label
    if label.startswith(("B-C-", "I-C-", "B-R-", "I-R-")):
        prefix, role = label[:2], label[4:]
        mapped = f"{prefix}{role}"
        if mapped in SRL_TAGS_SET: return mapped
    if "ARG5" in label: return label.replace("ARG5", "ARG4")
    if "ARG1-DSP" in label: return label.replace("ARG1-DSP", "ARG1")
    return "O"

# Load full NER dataset
with open("data/prepared/kniv-deberta-cascade/ner_spanmarker_train.json") as f:
    all_ner = json.load(f)
print(f"  Loaded {len(all_ner)} examples")

all_srl = []
batch_size = 64
for i in range(0, len(all_ner), batch_size):
    batch = all_ner[i:i+batch_size]
    texts = [ex["text"] for ex in batch]
    encoding = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    model_input = {k: v.to(device) for k, v in encoding.items()}
    with torch.no_grad():
        outputs = model(**model_input)
    preds = outputs.logits.argmax(dim=-1).cpu()

    for j, ex in enumerate(batch):
        words = ex["words"]
        pred_labels = [id2label[p.item()] for p in preds[j]]
        word_ids = encoding.word_ids(j)
        bio_tags = ["O"] * len(words)
        prev_word = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev_word and wid < len(words):
                bio_tags[wid] = map_srl_label(pred_labels[k])
            prev_word = wid
        pred_idx = next((k for k, t in enumerate(bio_tags) if t == "V"), None)
        non_o = sum(1 for t in bio_tags if t not in ("O", "V"))
        if pred_idx is not None and non_o > 0:
            all_srl.append({
                "words": words,
                "text": ex["text"],
                "srl_tags": bio_tags,
                "predicate_idx": pred_idx,
            })

    if (i // batch_size) % 200 == 0:
        print(f"  {min(i+batch_size, len(all_ner))}/{len(all_ner)} → {len(all_srl)} valid", flush=True)

print(f"\n  Silver SRL: {len(all_srl)} from {len(all_ner)}")

# Load PropBank gold and combine
with open("data/prepared/kniv-deberta-cascade/srl_train.json") as f:
    old_srl = json.load(f)
gold = [ex for ex in old_srl if "predicate_idx" in ex and len(ex.get("srl_tags", [])) > 0]
# Keep PropBank gold (first 8690 from original)
propbank_gold = gold[:8690]

srl_train = propbank_gold + all_srl
import random
random.shuffle(srl_train)

with open("data/prepared/kniv-deberta-cascade/srl_full_train.json", "w") as f:
    json.dump(srl_train, f)
print(f"  Saved srl_full_train.json: {len(srl_train)} (PropBank {len(propbank_gold)} + Silver {len(all_srl)})")

del model
torch.cuda.empty_cache()
print("Done.")

In [ ]:
# Cell 18: Upload srl_full_train.json + retrain SRL MLP on 147K
from huggingface_hub import HfApi
import os

api = HfApi()
path = "data/prepared/kniv-deberta-cascade/srl_full_train.json"
size = os.path.getsize(path) / 1024 / 1024
print(f"Uploading srl_full_train.json ({size:.1f} MB)...")
api.upload_file(
    path_or_fileobj=path,
    path_in_repo="prepared/kniv-deberta-cascade/srl_full_train.json",
    repo_id="dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
)
print("Uploaded.")

os.chdir("/content/kniv-nlp-models")
os.system("git pull")

# Stage 3m: SRL MLP on full 147K (frozen encoder+POS+NER+DEP)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 3m \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage3/best

In [ ]:
# Cell 19: Save ALL checkpoints to Drive
import shutil, os

drive_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en"
os.makedirs(drive_dir, exist_ok=True)

for stage in ["stage2e", "stage2f", "stage3", "stage3m", "stage3s"]:
    src = f"outputs/kniv-deberta-cascade-large-nlp-en/{stage}/best"
    if os.path.exists(src):
        shutil.copytree(src, f"{drive_dir}/{stage}", dirs_exist_ok=True)
        print(f"Saved {stage}")
    else:
        print(f"  {stage} not found locally (may already be on Drive)")

# Also save any data files generated on Colab
for f in ["srl_full_train.json", "ner_spanmarker_train.json", "ner_spanmarker_dev.json", "ud_spanmarker_train.json"]:
    src = f"data/prepared/kniv-deberta-cascade/{f}"
    if os.path.exists(src):
        dst = f"{drive_dir}/data/{f}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / 1024 / 1024
        print(f"Saved data/{f} ({size:.1f} MB)")

print("\nAll saved. Safe to shutdown.")

In [ ]:
# Cell 6: Save all checkpoints to Drive
import shutil, os

drive_dir = "/content/drive/MyDrive/kniv-models/kniv-deberta-cascade-large-nlp-en"
os.makedirs(drive_dir, exist_ok=True)

for stage in ["stage2a", "stage2b", "stage2c", "stage2d", "stage3", "stage3s", "stage4", "stage5"]:
    src = f"outputs/kniv-deberta-cascade-large-nlp-en/{stage}/best"
    if os.path.exists(src):
        shutil.copytree(src, f"{drive_dir}/{stage}", dirs_exist_ok=True)
        print(f"Saved {stage}")

print("\nAll checkpoints saved to Drive.")

In [ ]:
# Cell 7: Stage 3 — DEP head (frozen encoder+POS+NER, 5 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 3 \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage2d/best

In [ ]:
# Cell 8: Stage 3s — SRL head (frozen encoder+POS+NER+DEP, 5 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 3s \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage3/best

In [ ]:
# Cell 9: Stage 4 — CLS head (frozen encoder+POS+NER+DEP+SRL, 5 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 4 \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage3s/best

In [ ]:
# Cell 10: Stage 5 — Joint fine-tune all heads (3 epochs)
!python models/kniv-deberta-cascade-large-nlp-en/staged_train.py --stage 5 \
    --checkpoint outputs/kniv-deberta-cascade-large-nlp-en/stage4/best

In [ ]:
# Cell 11: Pull latest code + data (use instead of Cell 2 when repo already cloned)
import os
os.chdir("/content/kniv-nlp-models")
!git pull

# Re-download data (picks up any new files)
from huggingface_hub import snapshot_download
snapshot_download(
    "dragonscale-ai/kniv-corpus-en",
    repo_type="dataset",
    allow_patterns=["prepared/kniv-deberta-cascade/*"],
    local_dir="/content/kniv-data",
)
print("Code and data updated.")